### self 검증 RAG 테스트
1. retrieve : 연관 문서 로드
2. check doc relevance : 가져온 문서와 질문과 연관성 검증
    1. 연관성이 있는 경우 : generate
    2. 연관성이 없는 경우 : END
3. generate : 답변 생성
4. check hallucination : 가져온 문서에서 생성한 답변인지 검증
    1. 환각증상인 경우 : generate
    2. 환각증상 없이 생성된 경우 : check helpfulness
5. check helpfulness : 답변이 질문에 대해 유용한 결과인지 검증
    1. 올바른 답변인 경우 : END
    2. 틀린 답변인 경우 : rewrite
6. rewrite : 질문을 사전을 바탕으로 재생성
    1. rewrite > retrieve 실행. (1부터 실행)



In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="bge-m3")

database = Chroma(
    persist_directory="./chroma-markdown-tax-bge-m3",
    collection_name="chroma_markdown_tax_bge_m3",
    embedding_function=embeddings,
)

In [ ]:
retriever = database.as_retriever(search_kwargs={"k": 3})

In [ ]:
from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    question: str
    context: List[Document]
    answer: str


graph_builder = StateGraph(AgentState)

### Retrieve

In [ ]:
def retrieve(state: AgentState) -> AgentState:
    question = state['question']

    print(f'retrieve question : {question}')         # 값 확인

    docs = retriever.invoke(question)
    return {'context' : docs}

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:3b", num_predict=200)

### Generate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

#langsmith hub > rlm/rag-prompt
generate_prompt = ChatPromptTemplate.from_template(f"""
당신은 질문-답변 작업을 위한 어시스턴트입니다. 다음에 제공된 검색된 문서를 사용하여 질문에 답하세요. 반드시 두 문장 이내로 답변하세요.
"정보가 없습니다" 또는 "제공되지 않습니다" 라고 답변하지 마세요.
계산이 필요한 질문은 반드시 직접 계산하여 숫자로 답변하세요.
질문: {{question}} 
문서: {{context}} 
답변:
""")

In [ ]:
def generate(state: AgentState) -> AgentState:
  question = state['question']

  print(f'generate : {question}')         # 값 확인

  context = state['context']
  rag_chain = generate_prompt | llm
  response = rag_chain.invoke({'question': question, 'context': context})
  
  print(response)

  return {'answer': response}

### Generate Test

In [ ]:
import time

# 중간 테스트
start = time.time()

question = '연간 소득이 5,000만원인 거주자가 납부해야 하는 종합소득세는 얼마인가요?'
context = retriever.invoke(question)
generate_state = {'question': question, 'context': context}
answer = generate(generate_state)

print(f"generate 응답시간: {time.time() - start:.2f}초")

### check_doc_relevance
#### langsmith hub > langchain-ai/rag-document-relevance
#### Prompt History
- 네트워크 이슈로 langchain-ai/rag-document-relevance 직접 사용 못해서 프롬프트 템플릿으로 변경
- 원문 rag-document-relevance 내용에서 score 리턴 내용이 없어서 텍스트 리턴에서 점수 리턴으로 변경. (hub에서 가져다 쓰면 score값 제공)
```
Output ONLY a JSON object like this:
{{"score": "1"}} if any fact is relevant
{{"score": "0"}} if all facts are irrelevant

No explanation. No other text. JSON only.
```
- return score 값이 자꾸 0으로 나와, 스텝별로 검증 이유 출력함.
```
Provide the final score and explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct.
Avoid simply stating the correct answer at the outset.
```
- 검증 이유에서는 FACT별로 점수가 1로 나오지만, 최종 점수에 대해서는 0으로 출력함. > llm모델 이슈 확인
- 모델을 변경 전에 프롬프트 재검토 (사용한 모델 : ollama qwen2.5:3b > 7b 변경 추천 받음)
- 프롬프트 변경 > 모델이 이해하기에는 프롬프트가 길고 복잡함. 우선 짧고 명료한 프롬프트로 변경
```
Is the FACT related to the QUESTION? 
Reply with ONLY one of these two options, nothing else:
{{"score": "1"}}
{{"score": "0"}}
```
- 결과 : 완전 엉뚱한 대답에 대해서는 0, 관련 있으면 1 출력함.

- json parser 보다 string 1/0 이 나을 것 같아서 stringParser 로 변경

In [ ]:
rag_document_relevance_system_prompt = """Is the FACT related to the QUESTION?
Reply with ONLY one of these two options, nothing else:
If the FACT is related related to the QUESTION, respond with "1",
If the FACT is completely unrelated to the QUESTION, respond with "0"
"""

rag_document_relevance_human_prompt = """FACTS: {documents}
QUESTION: {question}"""

doc_relevance_prompt = ChatPromptTemplate.from_messages([
  ("system", rag_document_relevance_system_prompt),
  ("human", rag_document_relevance_human_prompt)
])

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from typing import Literal

#주어진 state를 기반으로 문서의 관련성을 판단
def check_doc_relevance(state: AgentState) -> Literal['relevant', 'irrelevant']:
  question = state['question']
  context = state['context'] 

  print(f'check_doc_relevance question : {question}')
  print(f'check_doc_relevance context : {context}')

  doc_relevance_chain = doc_relevance_prompt | llm | StrOutputParser()
  response = doc_relevance_chain.invoke({'question' : question, 'documents' : context})

  print(f'check_doc_relevance response : {response}')

  if response == '1':
    return 'relevant'
  
  return 'irrelevant'

### check_doc_relevance Test

In [ ]:
# 중간 테스트
question = '연간 소득이 5,000만원인 거주자가 납부해야 하는 종합소득세는 얼마인가요?'
context = retriever.invoke(question)
check_doc_state = {'question': question, 'context': context}
answer = check_doc_relevance(check_doc_state)
print(answer)

### rewrite

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

dictionary = ['사람과 관련된 표현 -> 거주자']
rewrite_prompt = PromptTemplate.from_template(f"""
사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
만약 변경할 필요가 없다고 판단된다면, 사용자의 질문만 리턴해주세요.
사전: {dictionary}                                           
질문: {{question}}
""")

def rewrite(state: AgentState) -> AgentState:
  question = state['question']
  
  print(f'rewrite : {type(question)}')   # 타입 확인
  print(f'rewrite : {question}')         # 값 확인

  rewrite_chain = rewrite_prompt | llm | StrOutputParser()
  response = rewrite_chain.invoke(question)

  print(f'rewrite : {type(response)}')   # 타입 확인
  print(f'rewrite : {response}')         # 값 확인

  return {'question' : response}

### check_hallucination

In [ ]:
hallucination_prompt = PromptTemplate.from_template("""
You are grading a student's answer based on the given documents.
Reply with ONLY "not hallucinated" or "hallucinated". Nothing else.
- "not hallucinated": the answer is based on or calculated from the documents.
- "hallucinated": the answer contains information not found in the documents.                     
                                                    
documents: {documents}
student_answer: {student_answer}""")

hallucination_llm = ChatOllama(model="qwen2.5:3b", temperature=0)                                               

def check_hallucination(state: AgentState) -> Literal['hallucinated', 'not hallucinated']:
  answer = state['answer']
  context = state['context']
  context = [doc.page_content for doc in context]
  hallucination_chain = hallucination_prompt | hallucination_llm | StrOutputParser()
  response = hallucination_chain.invoke({'student_answer' : answer, 'documents' : context})
  
  print(f'check_hallucination response : {response}')
  
  return response

### check_hallucination Test

In [ ]:
question = '연간 소득이 5,000만원인 거주자가 납부해야 하는 종합소득세는 얼마인가요?'
context = retriever.invoke(question)
hallucination_state = {'answer': '연봉 5천만 원인 거주자의 소득세는 624만 원입니다.', 'context': context}
hallucination_result = check_hallucination(hallucination_state)
print(f'Hallucination Check: {hallucination_result}')

### check_helpfulness

In [ ]:
rag_answer_helpfulness_system_prompt = """You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Reply with ONLY one of these two options, nothing else:
If the STUDENT ANSWER is helpful to the QUESTION, respond with "helpful",
If the STUDENT ANSWER is not helpful to the QUESTION, respond with "unhelpful".
"""

rag_answer_helpfulness_human_prompt = """STUDENT ANSWER: {student_answer}
QUESTION: {question}"""

helpfulness_prompt = ChatPromptTemplate.from_messages([
  ("system", rag_answer_helpfulness_system_prompt),
  ("human", rag_answer_helpfulness_human_prompt)
])

In [ ]:
def check_helpfulness(state: AgentState) -> Literal['helpful', 'unhelpful']:
  answer = state['answer']
  question = state['question']

  helpfulness_chain = helpfulness_prompt | llm | StrOutputParser()
  response = helpfulness_chain.invoke({'student_answer' : answer, 'question' : question})
  
  print(f'check_helpfulness response : {response}')

  return response

def check_helpfulness_empty(state: AgentState) -> AgentState:
  return state

### check_helpfulness Test

In [ ]:
helpfulness_state = {'answer': '5,000만원 초과 금액의 세율은 24% 입니다.', 'question': question}
helpfulness_result = check_helpfulness(helpfulness_state)
print(f'Helpfulness Check: {helpfulness_result}')

### graph

In [ ]:
graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate', generate)
graph_builder.add_node('rewrite', rewrite)
graph_builder.add_node('check_helpfulness_empty', check_helpfulness_empty)

In [ ]:
from langgraph.graph import START, END

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_conditional_edges('retrieve', check_doc_relevance, {'relevant' : 'generate', 'irrelevant' : END})
graph_builder.add_conditional_edges('generate', check_hallucination, {'not hallucinated' : 'check_helpfulness_empty', 'hallucinated' : 'generate'})
graph_builder.add_conditional_edges('check_helpfulness_empty', check_helpfulness, {'helpful' : END, 'unhelpful' : 'rewrite'})
graph_builder.add_edge('rewrite', 'retrieve')

In [ ]:
graph = graph_builder.compile()

In [ ]:
initial_state = {'question': '연간 소득이 5,000만원인 거주자가 납부해야 하는 종합소득세는 얼마인가요?'}
graph.invoke(initial_state)